<a href="https://colab.research.google.com/github/LeonardooAlves/LSE-ST445/blob/main/Week%205/2_st445_week5_lab2_mongodb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ST445 Week 5: CRUD in MongoDB with PyMongo

**A self-guided notebook · about 90 minutes · follows Part 2 of the Week 5 lecture, and mirrors the SQLite notebook**

In the SQLite notebook you stored countries in **tables** of rows and columns. Here you store the same data in **MongoDB**, a **document database**, and carry out the same four **CRUD** operations:

| Operation | SQL (SQLite notebook) | MongoDB (this notebook) |
|---|---|---|
| **C**reate | `INSERT` | `insert_one`, `insert_many` |
| **R**ead | `SELECT ... WHERE ...` | `find_one`, `find` |
| **U**pdate | `UPDATE ... SET ... WHERE ...` | `update_one`, `update_many` |
| **D**elete | `DELETE FROM ... WHERE ...` | `delete_one`, `delete_many` |

### How each operation is introduced

As in the lecture and the SQLite notebook, every operation appears in three steps:

| Step | What you see |
|---|---|
| **General form** | the method's pattern, with placeholders such as `{filter}` that you replace with your own values |
| **Example** | the method written out for our data, and run |
| **Result** | what came back, explained in the Markdown cell underneath |

**Reading the general forms:** `{placeholder}` means "put your own value here"; `[ ... ]` means optional.

### Cell marks

🟢 **run and read** · 🟡 **fill the gaps** marked `____` · 🔵 **write it yourself** · ✍️ **write a short answer**. Exercise cells are followed by a check cell that prints ✅ or ❌.

**Before you start:** **File → Save a copy in Drive**, so your work is kept.

---
## 0. Setting up

### What MongoDB is, in four words

| Term | Meaning | SQL equivalent |
|---|---|---|
| **database** | a container for collections | a database |
| **collection** | a group of documents | a table, but **without a fixed set of columns** |
| **document** | one record, written like a Python dictionary, which can contain lists and other dictionaries | a row, but **nested** |
| **field** | one named value inside a document | a column |

Unlike SQLite, MongoDB is a **server**: Python connects to it over the network. **PyMongo** is the Python library that does this.

### Two ways to run this notebook

1. **With MongoDB Atlas** (recommended): a free cloud MongoDB server. You need a **connection string** (it starts `mongodb+srv://...`).
2. **Without a server:** the notebook falls back to **mongomock**, a small library that imitates MongoDB inside Python, with the same commands. Everything in this notebook works with it, so you can start straight away.

### Getting an Atlas connection string (once)

1. Create a free account at **mongodb.com/atlas** (Get Started > Sign in) using your LSE email, and create a free (M0) cluster. Any cloud provider and region will do.
2. Under **Database Access**, add a database user with a username and a password. Use a password you do not use anywhere else.
3. Under **Network Access**, add the IP address `0.0.0.0/0` (allow access from anywhere): Colab's machines change address every session. This is acceptable for a teaching cluster with no personal data; do not do it for real data.
4. Under **Connect → Drivers → Python**, copy the connection string, and replace `<password>` in it with your user's password.

### Keep the connection string secret

The connection string **contains your password**. Never paste it into a code cell, and never commit it to GitHub (Week 2: keys and secrets). Store it in **Colab Secrets** instead:

1. Click the **key icon** in Colab's left sidebar.
2. **Add new secret**, name it `MONGO_URI`, and paste the connection string as its value.
3. Switch on **Notebook access** for this notebook.

In [ ]:
# 🟢 Install the libraries: pymongo talks to MongoDB; mongomock imitates it without a server
%pip install --quiet "pymongo[srv]" mongomock

**What you should see:** a few lines of installation messages, or nothing at all if the libraries are already installed.

Now connect. The cell below reads `MONGO_URI` from Colab Secrets. If it is there, it connects to Atlas and **pings** the server to check the connection works; if not, it uses mongomock. Read the `if` / `else` structure: only one of the two branches runs.

In [ ]:
# 🟢 Connect to Atlas if a connection string is stored in Secrets; otherwise use mongomock
from pprint import pprint                       # prints dictionaries in a readable layout
from datetime import datetime, timezone

try:
    from google.colab import userdata
    MONGO_URI = userdata.get("MONGO_URI")       # the secret, if it exists
except Exception:
    MONGO_URI = None                            # not in Colab, or no secret stored

if MONGO_URI:
    from pymongo import MongoClient
    from pymongo.errors import DuplicateKeyError
    client = MongoClient(MONGO_URI)
    client.admin.command("ping")                # raises an error if the connection fails
    print("Connected to MongoDB Atlas")
else:
    import mongomock
    from mongomock import DuplicateKeyError
    client = mongomock.MongoClient()
    print("No MONGO_URI found: using mongomock, an in-memory imitation of MongoDB")

**What you should see:** either `Connected to MongoDB Atlas` or the mongomock message. If Atlas fails, the error message usually names the cause: a wrong password (`bad auth`), or the IP address not allowed (a timeout).

### Choose a database and a collection

Square brackets select a database from the client, and a collection from the database. **Nothing needs to be created first**: MongoDB creates the database and the collection the moment you insert the first document. (In SQL you had to run `CREATE TABLE` first.)

We **drop** (delete) the collection first, so that you can re-run the notebook from the top and always start empty.

In [ ]:
# 🟢 Select a database and a collection
db = client["st445"]                # the database called st445
countries = db["countries"]         # the collection called countries

countries.drop()                    # start empty: delete the collection if it exists
print("Collections in st445:", db.list_collection_names())

**What you should see:** `Collections in st445: []`: an empty list, because the collection was just dropped and nothing has been inserted yet.

In [ ]:
# 🟢 A helper for the exercises
def check(condition, hint="Not yet: read the task again."):
    """Print a tick if condition is True, otherwise a cross and a hint."""
    if condition:
        print("✅ Correct")
    else:
        print("❌", hint)

---
## 1. Documents and BSON types

A **document** is written as a Python **dictionary**: field names in quotes, a colon, then the value. Values can be numbers, text, true/false, dates, **lists** (arrays) and **other dictionaries** (sub-documents):

```python
{
    "_id": "GBR",                                   # text: the document's unique identifier
    "name": "United Kingdom",                        # text
    "population_m": 68.3,                           # a decimal number
    "joined_un": 1945,                              # a whole number
    "is_island": True,                              # true/false
    "languages": ["English", "Welsh"],              # a list (array)
    "capital": {"name": "London", "population_m": 9.0},   # a sub-document
    "indicators": [{"year": 2022, "gdp_pc_usd": 48900}],  # a list of sub-documents
    "updated_at": datetime(2026, 10, 5, tzinfo=timezone.utc)   # a date and time, in UTC
}
```

MongoDB stores documents as **BSON** ("binary JSON"). BSON adds types that JSON lacks (Week 1): Python's `int`, `float`, `str`, `bool`, `list`, `dict`, `None` and **`datetime`** are each stored as their own BSON type. **Dates are a real type** in MongoDB, not text as in SQLite.

### The `_id` field

Every document has a field called **`_id`**: its **primary key**, unique within the collection. You can choose it yourself (here, the ISO3 code). If you leave it out, MongoDB creates an **`ObjectId`**, a 12-byte identifier that is unique and roughly ordered by creation time.

---
## 2. Create: `insert_one` and `insert_many`

### 2.1 Insert one document

**General form**

```python
result = {collection}.insert_one({document})
result.inserted_id          # the _id of the new document
```

In [ ]:
# 🟢 Insert one country, with nested data
uk = {
    "_id": "GBR",
    "name": "United Kingdom",
    "income_group": "High income",
    "languages": ["English", "Welsh"],
    "capital": {"name": "London", "population_m": 9.0},
    "indicators": [
        {"year": 2021, "gdp_pc_usd": 46500},
        {"year": 2022, "gdp_pc_usd": 48900},
    ],
    "updated_at": datetime(2026, 10, 5, 14, 30, tzinfo=timezone.utc),
}

result = countries.insert_one(uk)
print("Inserted document with _id:", result.inserted_id)

**What you should see:** `Inserted document with _id: GBR`. The collection `countries` now exists, because the first document has been inserted.

**Reading the code:** the document is built first as a Python dictionary called `uk`, over several lines so each field is easy to read; then `insert_one(uk)` sends it to MongoDB. The method returns a **result object**; its attribute `inserted_id` holds the new document's `_id`.

### 2.2 Insert several documents

**General form**

```python
result = {collection}.insert_many([{document}, {document}, ...])
result.inserted_ids         # a list of the new documents' _ids
```

The documents go in a **list**: square brackets, separated by commas. Notice that the documents **do not need the same fields**: Qatar has no `indicators` and no `languages`, and France has no `capital`. A collection has no fixed schema (lecture: flexible schemas).

In [ ]:
# 🟢 Insert four more countries at once
others = [
    {
        "_id": "FRA",
        "name": "France",
        "income_group": "High income",
        "languages": ["French"],
        "indicators": [
            {"year": 2021, "gdp_pc_usd": 43700},
            {"year": 2022, "gdp_pc_usd": 44400},
        ],
    },
    {
        "_id": "IND",
        "name": "India",
        "income_group": "Lower middle income",
        "languages": ["Hindi", "English"],
        "capital": {"name": "New Delhi", "population_m": 33.8},
        "indicators": [
            {"year": 2021, "gdp_pc_usd": 2250},
            {"year": 2022, "gdp_pc_usd": 2410},
        ],
    },
    {
        "_id": "NGA",
        "name": "Nigeria",
        "income_group": "Lower middle income",
        "languages": ["English", "Hausa", "Yoruba"],
        "indicators": [
            {"year": 2022, "gdp_pc_usd": 2160},
        ],
    },
    {
        "_id": "QAT",
        "name": "Qatar",
        "income_group": "High income",
    },
]

result = countries.insert_many(others)
print("Inserted:", result.inserted_ids)
print("Documents in the collection:", countries.count_documents({}))

**What you should see:** `Inserted: ['FRA', 'IND', 'NGA', 'QAT']` and `Documents in the collection: 5`. `count_documents({})` counts the documents matching a filter; the empty filter `{}` matches every document (more on filters in section 3).

### 2.3 Letting MongoDB choose the `_id`

Leave `_id` out, and MongoDB creates an `ObjectId`. We use a separate collection for this demonstration, so that `countries` keeps its readable codes.

In [ ]:
# 🟢 A document without an _id
notes = db["notes"]
notes.drop()

result = notes.insert_one({"text": "Remember to check the units", "created_at": datetime.now(timezone.utc)})
print("The _id MongoDB created:", result.inserted_id)
print("Its type:", type(result.inserted_id).__name__)

**What you should see:** a 24-character hexadecimal value such as `6720f1c2a8b4e5d3c1a0b9f7`, of type `ObjectId`. Yours will differ: each one is unique.

### 2.4 `_id` must be unique

`_id` is the primary key, so inserting a second document with the same `_id` is **rejected**, as a duplicate primary key was in SQLite. `try` / `except` catches the error so the notebook keeps running.

In [ ]:
# 🟢 A duplicate _id is rejected
duplicate = {"_id": "GBR", "name": "Great Britain"}

try:
    countries.insert_one(duplicate)
except DuplicateKeyError as err:
    print("Rejected: duplicate _id")
    print("Message:", str(err)[:90])

**What you should see:** `Rejected: duplicate _id`, and a message containing `duplicate key error` and the `_id` value. The original United Kingdom document is unchanged.

**Other rules.** Apart from `_id`, MongoDB enforces **no** rules by default: no `NOT NULL`, no `CHECK`, no foreign keys. Two ways to add some:

* a **unique index** on another field (the next cell), which rejects duplicates in that field;
* a **schema validator** (`$jsonSchema`), which checks fields and types on every insert; it is set when the collection is created on a real server, and is beyond this notebook (see the MongoDB Manual, "Schema Validation").

In [ ]:
# 🟢 A unique index: no two countries may have the same name
countries.create_index("name", unique=True)

try:
    countries.insert_one({"_id": "XXX", "name": "France"})
except DuplicateKeyError:
    print("Rejected: a second document called France")

**What you should see:** `Rejected: a second document called France`. The unique index works like SQL's `UNIQUE` constraint.

### 🟡 Exercise 1: create

Insert **Brazil**: `_id` `"BRA"`, name `"Brazil"`, income group `"Upper middle income"`, languages `["Portuguese"]`, and one indicator for 2022 with `gdp_pc_usd` 10300. Fill every `____`.

In [ ]:
brazil = {
    "_id": ____,
    "name": ____,
    "income_group": ____,
    "languages": [____],
    "indicators": [
        {"year": ____, "gdp_pc_usd": ____},
    ],
}

result = countries.____(brazil)
print(result.inserted_id)

In [ ]:
# Check for exercise 1
doc = countries.find_one({"_id": "BRA"})
check(doc is not None and doc.get("indicators") == [{"year": 2022, "gdp_pc_usd": 10300}] and doc.get("languages") == ["Portuguese"],
      "Insert one document with _id 'BRA' and the fields as described.")

---
## 3. Read: `find_one` and `find`

### 3.1 One document: `find_one`

**General form**

```python
{collection}.find_one({filter})        # the first matching document, or None
```

A **filter** is a dictionary of conditions, like SQL's `WHERE`. `{"field": value}` means "the field equals this value".

In [ ]:
# 🟢 One document, found by its _id
doc = countries.find_one({"_id": "FRA"})
pprint(doc)

**What you should see:** France's whole document, printed by `pprint` with one field per line. Because a document is a Python dictionary, you read a field with square brackets:

In [ ]:
# 🟢 Reading fields from the returned dictionary
print(doc["name"])                     # a top-level field
print(doc["languages"][0])             # the first item of a list
print(doc["indicators"][1]["gdp_pc_usd"])   # the second indicator's GDP

missing = countries.find_one({"_id": "ESP"})
print(missing)                         # no match

**What you should see:** `France`, `French`, `44400`, then `None`: `find_one` returns `None` when nothing matches, rather than raising an error.

### 3.2 Many documents: `find`, and the cursor

**General form**

```python
{collection}.find({filter}, {projection})
```

`find` does not return the documents straight away. It returns a **cursor**: an object you **loop over** to get the documents one at a time. A `for` loop is the clearest way.

In [ ]:
# 🟢 Every high-income country, one at a time
cursor = countries.find({"income_group": "High income"})

for doc in cursor:
    print(doc["_id"], doc["name"])

**What you should see:** GBR, FRA and QAT, with their names.

**About cursors.** A cursor can be looped over **once**. To keep the results, turn the cursor into a list with `list(...)`; to analyse them, turn the list into a pandas DataFrame:

In [ ]:
# 🟢 From cursor to list to DataFrame
import pandas as pd

cursor = countries.find({"income_group": "High income"})
documents = list(cursor)                # all the matching documents, as a list
print(len(documents), "documents")

table = pd.DataFrame(documents)
table[["_id", "name", "income_group"]]

**What you should see:** `3 documents`, then a three-row table. Fields a document lacks (Qatar's `languages`, for example) appear as `NaN` in the DataFrame.

### 3.3 Projections: choosing the fields

The **projection**, the second argument of `find`, chooses which fields come back, like the column list after SQL's `SELECT`: `1` shows a field, `0` hides it. `_id` is shown unless you hide it.

In [ ]:
# 🟢 Only the name, without _id
cursor = countries.find({"income_group": "High income"}, {"name": 1, "_id": 0})

for doc in cursor:
    print(doc)

**What you should see:** three small dictionaries, each with only `name`: `{'name': 'United Kingdom'}`, and so on.

### 3.4 Query operators

A field can be compared with an **operator** instead of a single value. Operators start with `$`, and go inside a dictionary for the field:

**General form**

```python
{"{field}": {"{$operator}": {value}}}
```

| Operator | Meaning | SQL |
|---|---|---|
| `$eq`, `$ne` | equal, not equal | `=`, `<>` |
| `$gt`, `$gte` | greater than, greater or equal | `>`, `>=` |
| `$lt`, `$lte` | less than, less or equal | `<`, `<=` |
| `$in`, `$nin` | in a list, not in a list | `IN`, `NOT IN` |
| `$exists` | the field is present (`True`) or absent (`False`) | `IS NOT NULL`, `IS NULL` |
| `$regex` | a text pattern | `LIKE` |

In [ ]:
# 🟢 $in: one of several values
cursor = countries.find({"_id": {"$in": ["IND", "NGA"]}}, {"name": 1})

for doc in cursor:
    print(doc)

**What you should see:** India and Nigeria.

In [ ]:
# 🟢 $exists: countries that have no indicators field at all
cursor = countries.find({"indicators": {"$exists": False}}, {"name": 1})

for doc in cursor:
    print(doc)

**What you should see:** only Qatar. `$exists: False` finds documents where the field is **missing**, which in a flexible schema is different from a field present with the value `None`.

### 3.5 Combining conditions

Several fields in **one** filter must **all** be true: an implicit **AND**. For **OR**, use `$or` with a **list** of filters.

In [ ]:
# 🟢 AND: two conditions in one filter
cursor = countries.find(
    {"income_group": "High income", "languages": "French"},
    {"name": 1},
)
for doc in cursor:
    print("AND:", doc)

# 🟢 OR: $or with a list of filters
cursor = countries.find(
    {"$or": [{"_id": "QAT"}, {"languages": "Hausa"}]},
    {"name": 1},
)
for doc in cursor:
    print("OR: ", doc)

**What you should see:** France for the AND query (high income **and** French is a language); Nigeria and Qatar for the OR query.

**Arrays match element by element.** `{"languages": "French"}` matched France even though `languages` is a **list**: for an array field, the filter is true if **any** element matches. That is how one document can hold several values in a field without breaking anything, unlike the "English; Welsh" text in the SQLite notebook's 1NF example.

### 3.6 Inside nested documents: dot notation

To reach a field inside a sub-document or inside a list of sub-documents, join the field names with a **dot**, inside quotes:

In [ ]:
# 🟢 A field inside a sub-document
doc = countries.find_one({"capital.name": "London"}, {"name": 1})
print(doc)

# 🟢 A field inside a list of sub-documents, with an operator
cursor = countries.find({"indicators.gdp_pc_usd": {"$gt": 45000}}, {"name": 1})
for doc in cursor:
    print(doc)

**What you should see:** the United Kingdom twice: it has the capital London, and it is the only country with an indicator above 45,000 (48,900 in 2022).

### 3.7 Sorting, limiting and counting

Methods on the cursor work like SQL's `ORDER BY` and `LIMIT`. `1` sorts ascending, `-1` descending.

| Method | Meaning | SQL |
|---|---|---|
| `.sort("{field}", 1)` | sort ascending (`-1` for descending) | `ORDER BY ... [DESC]` |
| `.limit({n})` | the first `n` documents | `LIMIT n` |
| `.skip({n})` | skip the first `n` documents | `OFFSET n` |
| `count_documents({filter})` | how many match | `SELECT COUNT(*) ... WHERE` |
| `distinct("{field}")` | the different values of a field | `SELECT DISTINCT` |

In [ ]:
# 🟢 Sorted by name, first three
cursor = countries.find({}, {"name": 1, "_id": 0})
cursor = cursor.sort("name", 1)
cursor = cursor.limit(3)

for doc in cursor:
    print(doc["name"])

**What you should see:** Brazil, France, India: the first three names in alphabetical order. (The three steps could be written as one chain, `find(...).sort(...).limit(3)`; here they are on separate lines so each step is visible.)

In [ ]:
# 🟢 Counting, and the distinct values of a field
n_high = countries.count_documents({"income_group": "High income"})
print("High-income countries:", n_high)

groups = countries.distinct("income_group")
print("Income groups:", sorted(groups))

**What you should see:** `High-income countries: 3`, and the three income groups present: High income, Lower middle income, Upper middle income.

### 🟡 Exercise 2: read

1. Store in `english` the **names** of every country that lists English as a language, as a Python list. (Loop over the cursor and append each name.)
2. Store in `n_with_capital` the number of documents that **have** a `capital` field.

In [ ]:
# 1.
english = []
cursor = countries.find({"languages": ____}, {"name": 1})
for doc in cursor:
    english.append(doc[____])

# 2.
n_with_capital = countries.count_documents({"capital": {"____": True}})

print(english, n_with_capital)

In [ ]:
# Check for exercise 2
check(sorted(english) == ["India", "Nigeria", "United Kingdom"], "1. Filter on languages, then append doc['name'].")
check(n_with_capital == 2, "2. Use $exists: True on the capital field.")

---
## 4. Update: `update_one` and `update_many`

**General form**

```python
result = {collection}.update_one({filter}, {update})
result.matched_count       # how many documents matched the filter
result.modified_count      # how many were actually changed
```

The first argument chooses the document, exactly like a `find` filter. The second says **what to change**, using an **update operator**:

| Operator | Does | Example |
|---|---|---|
| `$set` | sets fields to values (adding them if missing) | `{"$set": {"region": "Europe"}}` |
| `$unset` | removes fields | `{"$unset": {"region": ""}}` |
| `$inc` | adds to a number | `{"$inc": {"visits": 1}}` |
| `$push` | adds an item to the end of a list | `{"$push": {"languages": "Gaelic"}}` |
| `$addToSet` | adds an item to a list only if it is not there | `{"$addToSet": {"languages": "Welsh"}}` |
| `$pull` | removes matching items from a list | `{"$pull": {"languages": "Gaelic"}}` |

**Look first.** As in SQL, run the filter with `find` before an update, and check the counts afterwards.

### 4.1 `$set`: change or add fields

In [ ]:
# 🟢 Look first: what will the filter match?
pprint(countries.find_one({"_id": "GBR"}, {"name": 1, "region": 1}))

**What you should see:** `{'_id': 'GBR', 'name': 'United Kingdom'}`: there is no `region` field yet.

In [ ]:
# 🟢 Then update, and check the counts
result = countries.update_one(
    {"_id": "GBR"},
    {"$set": {"region": "Europe", "updated_at": datetime.now(timezone.utc)}},
)
print("matched:", result.matched_count, "| modified:", result.modified_count)

pprint(countries.find_one({"_id": "GBR"}, {"name": 1, "region": 1}))

**What you should see:** `matched: 1 | modified: 1`, and the document now has `region: 'Europe'`. Every other field (languages, indicators) is untouched: `$set` changes **only** the fields it names.

**`matched` versus `modified`.** Run the same update twice and the second time `modified_count` is `0`: the document matched, but already had those values. A `matched_count` of `0` means your filter found nothing, which usually means a typo.

### 4.2 Lists: `$push` and `$addToSet`

In [ ]:
# 🟢 $push adds to the end of a list
countries.update_one({"_id": "GBR"}, {"$push": {"languages": "Scottish Gaelic"}})
print(countries.find_one({"_id": "GBR"})["languages"])

# 🟢 $addToSet adds only if the value is not already there
countries.update_one({"_id": "GBR"}, {"$addToSet": {"languages": "Welsh"}})
print(countries.find_one({"_id": "GBR"})["languages"])

**What you should see:** `['English', 'Welsh', 'Scottish Gaelic']` twice: `$push` added Scottish Gaelic; `$addToSet` did nothing, because Welsh was already in the list.

**Adding a year of data** is a `$push` too: the new indicator is a sub-document added to the `indicators` list.

In [ ]:
# 🟢 Add a 2023 indicator to France
result = countries.update_one(
    {"_id": "FRA"},
    {"$push": {"indicators": {"year": 2023, "gdp_pc_usd": 46300}}},
)
print("modified:", result.modified_count)

for item in countries.find_one({"_id": "FRA"})["indicators"]:
    print(item)

**What you should see:** `modified: 1`, then France's three yearly indicators, 2021, 2022 and 2023.

### 4.3 `$inc` and `$unset`

In [ ]:
# 🟢 $inc adds to a number (and creates the field, starting from 0, if it is missing)
countries.update_one({"_id": "FRA"}, {"$inc": {"times_reviewed": 1}})
countries.update_one({"_id": "FRA"}, {"$inc": {"times_reviewed": 1}})
print("times_reviewed:", countries.find_one({"_id": "FRA"})["times_reviewed"])

# 🟢 $unset removes a field (the value given, here "", is ignored)
countries.update_one({"_id": "FRA"}, {"$unset": {"times_reviewed": ""}})
print("still there?", "times_reviewed" in countries.find_one({"_id": "FRA"}))

**What you should see:** `times_reviewed: 2` (two increments from nothing), then `still there? False`.

### 4.4 `update_many`: every matching document

In [ ]:
# 🟢 Mark every high-income country as reviewed
result = countries.update_many(
    {"income_group": "High income"},
    {"$set": {"reviewed": True}},
)
print("matched:", result.matched_count, "| modified:", result.modified_count)

**What you should see:** `matched: 3 | modified: 3`: GBR, FRA and QAT. `update_one` would have changed only the first of them.

### 4.5 Upsert: update, or insert if missing

With `upsert=True`, an update that matches nothing **inserts** a new document instead, built from the filter and the update.

In [ ]:
# 🟢 Spain does not exist yet: upsert inserts it
result = countries.update_one(
    {"_id": "ESP"},
    {"$set": {"name": "Spain", "income_group": "High income"}},
    upsert=True,
)
print("matched:", result.matched_count, "| upserted _id:", result.upserted_id)
pprint(countries.find_one({"_id": "ESP"}))

**What you should see:** `matched: 0 | upserted _id: ESP`, and the new Spain document. Run the cell again and it **updates** Spain instead (`matched: 1`, `upserted _id: None`): upserts are useful when re-running a data collection script.

### 4.6 The trap: replacing a whole document

`replace_one` swaps the **entire** document for a new one, keeping only `_id`. It is occasionally what you want, and easy to use by mistake when you meant `$set`:

In [ ]:
# 🟢 replace_one keeps _id and replaces everything else
countries.replace_one({"_id": "ESP"}, {"name": "Spain"})
pprint(countries.find_one({"_id": "ESP"}))

**What you should see:** `{'_id': 'ESP', 'name': 'Spain'}`: the `income_group` field is **gone**. To change one field, always use `update_one` with `$set`. (PyMongo refuses an `update_one` whose second argument has no `$` operator, to protect you from exactly this mistake.)

### 🟡 Exercise 3: update

1. Add the field `"region": "Asia"` to India, and store the result's `modified_count` in `n_modified`.
2. Add the 2023 indicator `{"year": 2023, "gdp_pc_usd": 2480}` to India's `indicators` list.
3. With one `update_many`, set `"reviewed": False` on every country in the `"Lower middle income"` group, and store `matched_count` in `n_lmic`.

In [ ]:
# 1.
result = countries.update_one({"_id": "IND"}, {"____": {"region": ____}})
n_modified = result.____

# 2.
countries.update_one({"_id": "IND"}, {"____": {"indicators": {"year": 2023, "gdp_pc_usd": 2480}}})

# 3.
result = countries.____({"income_group": ____}, {"$set": {"reviewed": False}})
n_lmic = result.matched_count

In [ ]:
# Check for exercise 3
india = countries.find_one({"_id": "IND"})
check(n_modified == 1 and india.get("region") == "Asia", "1. $set region to 'Asia'; n_modified = result.modified_count.")
check({"year": 2023, "gdp_pc_usd": 2480} in india.get("indicators", []), "2. $push the new indicator onto indicators.")
check(n_lmic == 2 and countries.count_documents({"reviewed": False}) == 2, "3. update_many on Lower middle income (India and Nigeria).")

---
## 5. Delete: `delete_one` and `delete_many`

**General form**

```python
result = {collection}.delete_one({filter})     # the first match
result = {collection}.delete_many({filter})    # every match
result.deleted_count                           # how many were deleted
```

**Look first**, with the same filter in `find`, because a deleted document cannot be recovered.

In [ ]:
# 🟢 Look first: which documents have no indicators?
cursor = countries.find({"indicators": {"$exists": False}}, {"name": 1})
for doc in cursor:
    print(doc)

**What you should see:** Qatar and Spain: the two countries with no `indicators` field.

In [ ]:
# 🟢 Then delete them, and check the count
result = countries.delete_many({"indicators": {"$exists": False}})
print("deleted:", result.deleted_count)
print("left:", countries.count_documents({}))

**What you should see:** `deleted: 2` and `left: 5` (GBR, FRA, IND, NGA, BRA).

**The trap: an empty filter.** `delete_many({})` matches **every** document and empties the collection: the MongoDB version of SQL's `DELETE` without `WHERE`. It is demonstrated here on the `notes` collection, which we no longer need:

In [ ]:
# 🟢 An empty filter deletes everything
result = notes.delete_many({})
print("deleted from notes:", result.deleted_count)
print("left in notes:", notes.count_documents({}))

**What you should see:** `deleted from notes: 1`, `left in notes: 0`. The collection still exists, but empty. `notes.drop()` removes the collection itself.

### 🔵 Exercise 4: delete

Delete **Brazil**, using `delete_one` with a filter on its `_id`. Store the number deleted in `n_deleted`. Look first with `find_one` if you like.

In [ ]:
# Your code here

In [ ]:
# Check for exercise 4
check(n_deleted == 1 and countries.find_one({"_id": "BRA"}) is None and countries.count_documents({}) == 4,
      "delete_one({'_id': 'BRA'}), then n_deleted = result.deleted_count.")

---
## 6. Putting it together: one country, start to finish

Every CRUD operation once more, on one new document, with a check after each step.

In [ ]:
# Create
countries.insert_one({"_id": "KEN", "name": "Kenya", "income_group": "Lower middle income"})
print("1. created:", countries.find_one({"_id": "KEN"}))

# Read
doc = countries.find_one({"_id": "KEN"}, {"name": 1, "_id": 0})
print("2. read:   ", doc)

# Update
result = countries.update_one({"_id": "KEN"}, {"$set": {"languages": ["Swahili", "English"]}})
print("3. updated:", result.modified_count, "document;", countries.find_one({"_id": "KEN"})["languages"])

# Delete
result = countries.delete_one({"_id": "KEN"})
print("4. deleted:", result.deleted_count, "document; still there?", countries.find_one({"_id": "KEN"}) is not None)

**What you should see:** four numbered lines: Kenya created, its name read back, its languages added (`1 document`), and finally deleted (`still there? False`).

### Closing the connection

In [ ]:
client.close()
print("Connection closed")

### Summary: SQL and MongoDB, side by side

| Task | SQL (SQLite notebook) | MongoDB (this notebook) |
|---|---|---|
| Create a container | `CREATE TABLE country (...)` | nothing: the first insert creates the collection |
| Create | `INSERT INTO country VALUES (...)` | `countries.insert_one({...})`, `insert_many([...])` |
| Primary key | `iso3 TEXT PRIMARY KEY` | `_id` (an `ObjectId` if not given) |
| Read one | `SELECT * FROM country WHERE iso3 = 'FRA'` | `countries.find_one({"_id": "FRA"})` |
| Read with conditions | `WHERE gdp > 45000 AND ...` | `find({"gdp": {"$gt": 45000}, ...})` |
| Choose columns | `SELECT name FROM ...` | `find({...}, {"name": 1, "_id": 0})` |
| Sort and limit | `ORDER BY name LIMIT 3` | `.sort("name", 1).limit(3)` |
| Count | `SELECT COUNT(*) ... WHERE ...` | `count_documents({...})` |
| Update | `UPDATE ... SET region = 'Europe' WHERE ...` | `update_one({...}, {"$set": {"region": "Europe"}})` |
| Delete | `DELETE FROM ... WHERE ...` | `delete_one({...})`, `delete_many({...})` |
| Check what changed | `cursor.rowcount` | `matched_count`, `modified_count`, `deleted_count` |
| Nested data | separate tables and joins | lists and sub-documents, reached with dot notation |
| Rules | `NOT NULL`, `CHECK`, `FOREIGN KEY` | `_id` and unique indexes by default; schema validation if you add it |

**Next:** the lecture's aggregation pipeline (`$match`, `$group`, `$lookup`) builds on these filters; the seminar's second lab loads your cleaned Week 4 data into MongoDB and compares it with DuckDB.

**References.** MongoDB Manual: "Documents", "BSON Types", "Query and Projection Operators", "Update Operators", "Schema Validation" (mongodb.com/docs/manual); PyMongo documentation, "Tutorial" (pymongo.readthedocs.io); Bradshaw, S., Brazil, E. and Chodorow, K. (2019), *MongoDB: The Definitive Guide*, 3rd ed., O'Reilly, chapters 2 to 4.